# 02 Item2Vec Baseline: Books + Movies Intersection

## Setup

In [1]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
!git checkout item-2-vec && git pull
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.08 KiB | 137.00 KiB/s, done.
From https://github.com/mkobycheva/recommendation-system
   924780c..bb6bc39  item-2-vec -> origin/item-2-vec
Updating 924780c..bb6bc39
Fast-forward
 src/evaluation/metrics.py | 1 +
 1 file changed, 1 insertion(+)
/content/recommendation-system
Already on 'item-2-vec'
Your branch is up to date with 'origin/item-2-vec'.
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import pandas as pd
from gensim.models import Word2Vec

from src.evaluation.metrics import ndcg_at_k, recall_at_k

## Load Prepared Splits

In [3]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [4]:
def add_domain(df, domain):
    df = df.copy()
    df['domain'] = domain
    df['item_id'] = domain + '::' + df['parent_asin'].astype(str)
    return df

books_train = add_domain(books_train, 'books')
books_valid = add_domain(books_valid, 'books')
books_test = add_domain(books_test, 'books')

movies_train = add_domain(movies_train, 'movies')
movies_valid = add_domain(movies_valid, 'movies')
movies_test = add_domain(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

,user_id,item_id,domain
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0061713244,books
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0871139634,books
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0385521685,books
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0848732634,books
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0151014981,books


## Data prep for item2vec

In [5]:
# Building sequences for word2vec
# Each user -> list of his items (in order of interaction)
sequences = (
    train_df.sort_values('timestamp')
    .groupby('user_id')['item_id']
    .apply(list)
    .tolist()
)

print(f'Total sequences: {len(sequences)}')
print(f'Example: {sequences[0][:5]}')

Total sequences: 127188
Example: ['books::0061058386', 'books::0441008534', 'books::0441009239', 'movies::B0006B2A2O', 'books::0375826688']


## Word2Vec applied to Multi-Domain reommendation problem

In [6]:
model = Word2Vec(
    sentences=sequences,
    vector_size=128,
    window=5,
    sg=1,          # Skip-gram
    min_count=3,
    workers=4,
    seed=42,
)

print(f'Vocabulary size: {len(model.wv)}')
print(f'Example vector shape: {model.wv[sequences[0][0]].shape}')

Vocabulary size: 332487
Example vector shape: (128,)


# Recommendations

In [7]:
# Pre-compute domain item sets
domain_items = {
    domain: np.array([item for item in model.wv.index_to_key
                      if item.startswith(f'{domain}::')])
    for domain in ['books', 'movies']
}

print(f"Books in vocab:  {len(domain_items['books']):,}")
print(f"Movies in vocab: {len(domain_items['movies']):,}")

Books in vocab:  206,994
Movies in vocab: 125,493


In [8]:
# Pre-compute user -> train items dict
user_train_items = (
    train_df.groupby('user_id')['item_id']
    .apply(list)
    .to_dict()
)

In [9]:
def recommend_for_users(user_ids, target_domain, k=10, batch_size=5000):
    candidates = domain_items[target_domain]
    candidate_vectors = model.wv[candidates]
    candidate_to_idx = {item: i for i, item in enumerate(candidates)}
    recommendations = {}
    user_ids = list(user_ids)

    for batch_start in range(0, len(user_ids), batch_size):
        batch = user_ids[batch_start:batch_start + batch_size]
        user_vectors = []
        valid_user_ids = []
        known_items_list = []

        for user_id in batch:
            user_items = user_train_items.get(user_id, [])
            valid_items = [i for i in user_items if i in model.wv]
            if not valid_items:
                recommendations[user_id] = []
                continue
            user_vectors.append(np.mean(model.wv[valid_items], axis=0))
            valid_user_ids.append(user_id)
            known_items_list.append(set(user_items))

        if not user_vectors:
            continue

        user_matrix = np.stack(user_vectors)
        all_scores = user_matrix @ candidate_vectors.T

        for i, user_id in enumerate(valid_user_ids):
            scores = all_scores[i].copy()
            known_indices = np.array([candidate_to_idx[item] for item in known_items_list[i]
                                      if item in candidate_to_idx], dtype=np.int64)
            scores[known_indices] = -np.inf

            # argpartition is O(n) vs argsort O(n log n)
            top_indices = np.argpartition(-scores, k)[:k]
            top_indices = top_indices[np.argsort(-scores[top_indices])]
            recommendations[user_id] = [candidates[j] for j in top_indices]

        if batch_start % 10000 == 0:
            print(f"Processed {batch_start}/{len(user_ids)} users...")

    return recommendations


def relevant_items_by_user(split_df, target_domain):
    domain_rows = split_df[split_df['domain'] == target_domain]
    return domain_rows.groupby('user_id')['item_id'].agg(set).to_dict()

## Evaluate and Save Metrics

In [10]:
def evaluate_multi_domain(split_df, k=10):
    """
    Multi-domain: recommend items from BOTH domains simultaneously.
    Model is trained jointly on all domains (collective model).
    Evaluate each domain separately.
    """
    results = {}

    for target_domain in ['books', 'movies']:
        relevant = relevant_items_by_user(split_df, target_domain)
        eval_users = list(relevant.keys())

        print(f"Evaluating {target_domain}: {len(eval_users)} users")

        recs = recommend_for_users(eval_users, target_domain, k=k)

        ndcg_scores, recall_scores = [], []
        for user_id in eval_users:
            user_recs = recs.get(user_id, [])
            user_relevant = relevant.get(user_id, set())
            if not user_relevant:
                continue
            ndcg_scores.append(ndcg_at_k(user_recs, user_relevant, k))
            recall_scores.append(recall_at_k(user_recs, user_relevant, k))

        results[target_domain] = {
            'ndcg@10': round(np.mean(ndcg_scores), 4),
            'recall@10': round(np.mean(recall_scores), 4),
            'n_users': len(ndcg_scores),
        }

    return results

In [11]:
# Check on validation set
K = 10
valid_results = evaluate_multi_domain(valid_df, k=K)

for domain, metrics in valid_results.items():
    print(f"\n{domain}:")
    print(f"  NDCG@10:   {metrics['ndcg@10']}")
    print(f"  Recall@10: {metrics['recall@10']}")
    print(f"  Users:     {metrics['n_users']}")

Evaluating books: 127188 users
Processed 0/127188 users...
Processed 10000/127188 users...
Processed 20000/127188 users...
Processed 30000/127188 users...
Processed 40000/127188 users...
Processed 50000/127188 users...
Processed 60000/127188 users...
Processed 70000/127188 users...
Processed 80000/127188 users...
Processed 90000/127188 users...
Processed 100000/127188 users...
Processed 110000/127188 users...
Processed 120000/127188 users...
Evaluating movies: 127188 users
Processed 0/127188 users...
Processed 10000/127188 users...
Processed 20000/127188 users...
Processed 30000/127188 users...
Processed 40000/127188 users...
Processed 50000/127188 users...
Processed 60000/127188 users...
Processed 70000/127188 users...
Processed 80000/127188 users...
Processed 90000/127188 users...
Processed 100000/127188 users...
Processed 110000/127188 users...
Processed 120000/127188 users...

books:
  NDCG@10:   0.0012
  Recall@10: 0.0025
  Users:     127188

movies:
  NDCG@10:   0.0026
  Recall@1